# Merge Data ERA5 Pasuruan (2016-2025)

Notebook ini menggabungkan seluruh file NetCDF (`.nc`) bulanan ERA5 untuk wilayah Pasuruan (2016-2025) menjadi **satu file CSV** utuh.

**Tahapan:**
1. Import library yang dibutuhkan
2. Cari semua file `.nc` di folder data
3. Buka & gabungkan (merge) seluruh file menjadi satu dataset
4. Ubah dataset menjadi DataFrame (tabel)
5. Export ke CSV


In [1]:
# 1. Import library
import os
import glob
import pandas as pd
import xarray as xr


In [10]:
data_folder = r"C:\Data analyst, scient, ML\data_scienct\prediksi hujan pasuruan\dowload_dataset\era5_data"

file_list = sorted(glob.glob(os.path.join(data_folder, "*.nc")))

print(f"Jumlah file ditemukan: {len(file_list)}")
print(file_list[:5])

Jumlah file ditemukan: 120
['C:\\Data analyst, scient, ML\\data_scienct\\prediksi hujan pasuruan\\dowload_dataset\\era5_data\\era5_pasuruan_2016_01.nc', 'C:\\Data analyst, scient, ML\\data_scienct\\prediksi hujan pasuruan\\dowload_dataset\\era5_data\\era5_pasuruan_2016_02.nc', 'C:\\Data analyst, scient, ML\\data_scienct\\prediksi hujan pasuruan\\dowload_dataset\\era5_data\\era5_pasuruan_2016_03.nc', 'C:\\Data analyst, scient, ML\\data_scienct\\prediksi hujan pasuruan\\dowload_dataset\\era5_data\\era5_pasuruan_2016_04.nc', 'C:\\Data analyst, scient, ML\\data_scienct\\prediksi hujan pasuruan\\dowload_dataset\\era5_data\\era5_pasuruan_2016_05.nc']


## 3. Gabungkan seluruh file NetCDF

Karena setiap file berisi data bulanan dengan dimensi waktu, latitude, dan longitude yang sama,
kita gunakan `xr.open_mfdataset` untuk membaca dan menggabungkan semua file sekaligus berdasarkan dimensi `valid_time`.


In [15]:
pip install xarray netCDF4 dask h5netcdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\Yusufalqodri\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [18]:
import dask
print(dask.__version__)

2026.6.0


In [19]:
# 3. Buka dan gabungkan semua file .nc menjadi satu dataset
ds = xr.open_mfdataset(
    file_list,
    combine="by_coords",   # gabungkan otomatis berdasarkan koordinat (waktu)
    engine="netcdf4"
)

ds


ImportError: chunk manager 'dask' is not available. Please make sure 'dask' is installed and importable.

In [4]:
# Urutkan berdasarkan waktu agar data tersusun rapi dari 2016 sampai 2025
ds = ds.sortby("valid_time")

print("Rentang waktu data:", ds.valid_time.values[0], "sampai", ds.valid_time.values[-1])
print("Jumlah total timestep:", ds.sizes["valid_time"])


Rentang waktu data: 2016-01-01T00:00:00.000000000 sampai 2025-12-31T18:00:00.000000000
Jumlah total timestep: 14612


## 4. Ubah dataset menjadi DataFrame (tabel)

Dataset memiliki dimensi `valid_time`, `latitude`, dan `longitude`, dengan variabel:
- `tp`  : total precipitation (curah hujan)
- `t2m` : temperature 2m (suhu udara)
- `d2m` : dewpoint temperature 2m
- `u10` : komponen angin arah timur (10m)
- `v10` : komponen angin arah utara (10m)

Kita ubah ke bentuk tabel panjang (long format) agar setiap baris berisi 1 titik waktu + 1 titik koordinat.


In [5]:
# 4. Konversi dataset ke DataFrame
df = ds.to_dataframe().reset_index()

# Buang kolom bantu dari grib (expver, number) jika ada, karena tidak diperlukan
drop_cols = [c for c in ["expver", "number"] if c in df.columns]
df = df.drop(columns=drop_cols)

print("Jumlah baris:", len(df))
df.head()


Jumlah baris: 818272


,valid_time,latitude,longitude,tp,t2m,d2m,u10,v10
0,2016-01-01,-7.5,112.5,0.006236,299.883789,298.254150,0.148270,0.356661
1,2016-01-01,-7.5,112.6,0.005594,300.065430,298.037354,0.090164,0.305391
2,2016-01-01,-7.5,112.7,0.004490,300.147461,297.574463,0.054031,0.281466
3,2016-01-01,-7.5,112.8,NaN,NaN,NaN,NaN,NaN
4,2016-01-01,-7.5,112.9,NaN,NaN,NaN,NaN,NaN


In [6]:
# Rapikan nama kolom & urutan kolom
df = df.rename(columns={"valid_time": "datetime"})
df = df.sort_values(["datetime", "latitude", "longitude"]).reset_index(drop=True)

df.head()


,datetime,latitude,longitude,tp,t2m,d2m,u10,v10
0,2016-01-01,-8.1,112.5,0.003074,297.596680,295.574463,-0.440109,0.633028
1,2016-01-01,-8.1,112.6,0.003453,298.239258,296.211182,-0.356613,0.687716
2,2016-01-01,-8.1,112.7,0.003868,298.301758,296.260010,-0.234543,0.770723
3,2016-01-01,-8.1,112.8,0.004375,295.536133,293.476807,-0.135910,0.847383
4,2016-01-01,-8.1,112.9,0.005261,293.100586,291.168213,-0.149582,0.888887


## 5. Export ke CSV

Simpan seluruh data yang sudah digabung ke dalam satu file CSV utuh.


In [7]:
# 5. Simpan ke file CSV
output_path = "era5_pasuruan_2016-2025_merged.csv"
df.to_csv(output_path, index=False)

print(f"Berhasil disimpan: {output_path}")
print(f"Total baris: {len(df):,}")
print(f"Total kolom: {len(df.columns)}")


Berhasil disimpan: era5_pasuruan_2016-2025_merged.csv
Total baris: 818,272
Total kolom: 8


In [8]:
# Cek ulang hasil file CSV
check = pd.read_csv(output_path)
check.info()
check.head()


<class 'pandas.DataFrame'>
RangeIndex: 818272 entries, 0 to 818271
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   datetime   818272 non-null  str    
 1   latitude   818272 non-null  float64
 2   longitude  818272 non-null  float64
 3   tp         672152 non-null  float64
 4   t2m        672152 non-null  float64
 5   d2m        672152 non-null  float64
 6   u10        672152 non-null  float64
 7   v10        672152 non-null  float64
dtypes: float64(7), str(1)
memory usage: 49.9 MB


,datetime,latitude,longitude,tp,t2m,d2m,u10,v10
0,2016-01-01 00:00:00,-8.1,112.5,0.003074,297.59668,295.57446,-0.440109,0.633028
1,2016-01-01 00:00:00,-8.1,112.6,0.003453,298.23926,296.21118,-0.356613,0.687716
2,2016-01-01 00:00:00,-8.1,112.7,0.003868,298.30176,296.26000,-0.234543,0.770723
3,2016-01-01 00:00:00,-8.1,112.8,0.004375,295.53613,293.47680,-0.135910,0.847383
4,2016-01-01 00:00:00,-8.1,112.9,0.005261,293.10060,291.16820,-0.149582,0.888887
